## 嵌入

在构建 RAG 系统之前，您需要了解嵌入实际上是什么 - 因为它们是构建其他所有内容的基础
文本嵌入是将一段文本投影到高维向量空间中

该文本在此空间中的位置表示为一长串数字

至关重要的是，语义相似的文本最终在该空间中紧密结合在一起——这使得相似性搜索成为可能

资源：

1、 Stack Overflow 博客：文本嵌入的直观介绍（免费）

链接：<https://stackoverflow.blog/2023/11/09/an-intuitive-introduction-to-text-embeddings/>

最好的初学者解释。由一位花费多年构建 NLP 产品的开发人员编写，重点是构建正确的直觉而不是数学

2、 Google ML 速成课程：嵌入（免费）

链接：<https://developers.google.com/machine-learning/crash-course/embeddings>

涵盖为什么密集向量表示可以解决 one-hot 编码无法解决的问题 - 具体来说，捕获项目之间的语义关系

3、HuggingFace：嵌入入门（免费）

链接：<https://huggingface.co/blog/getting-started-with-embeddings>

动手指导。展示如何使用句子转换器库生成嵌入、托管它们以及使用它们对真实的常见问题解答数据集进行语义搜索

4、 OpenAI Embeddings Guide（官方文档，免费）

链接：<https://platform.openai.com/docs/guides/embeddings>

在代码中使用 OpenAI 的 text-embedding-3-small 和 text-embedding-3-large 模型的参考

**重点关注内容：** 向量的概念是什么、为什么相似的文本会产生相似的向量、余弦相似度如何工作、嵌入模型（OpenAI、HuggingFace 句子转换器）之间的差异以及嵌入维度在实践中意味着什么

**练习：** 取 20 个相关主题的句子，使用 OpenAI 或句子转换器嵌入它们，并编写一个简单的最近邻搜索，返回与查询最相似的 3 个句子。这实际上是 RAG 核心的缩影

### 向量是什么？

向量就是一组有方向和大小的数。

在数学里，它常写成这样：

```text
[2, 3]
[0.1, -0.7, 1.2]
```

这不只是“几个数字”，而是可以理解成空间里的一个点，或者一支箭头。

1、 **直观理解**

二维里：

```text
y
^
|        • (2,3)
|
|
+----------------> x
```
[2, 3] 表示从原点走：

- 向右 2
- 向上 3

2、**在 AI 里为什么老提向量**

因为模型不认识人类语言本身，它更擅长处理数字。所以我们会把文本、图片、音频，变成向量。

比如一句话：

```text
“我喜欢机器学习”
```

可能被编码成：

```text
[0.12, -0.31, 0.88, ..., 1.07]
```

这个向量就像这句话在高维空间里的“坐标”。

3、**向量在 embedding 里是什么意思**

在 embedding 里，向量不是让你看每个数字本身，而是看：

- 两个向量离得近不近
- 两个向量方向像不像

如果两句话意思接近：

- “机器学习怎么入门”
- “如何开始学习机器学习”

它们对应的向量通常也更接近。

#### 为什么相似的文本会产生相似的向量？

因为 embedding 模型在训练时，会被优化成把语义相近的文本放到高维空间中彼此靠近的位置。

例如：

- “机器学习怎么入门”
- “如何开始学习机器学习”

虽然字面不完全相同，但它们通常：

- 出现在相似上下文中
- 回答相似问题
- 和相似知识一起出现

模型在大量数据上学习后，就会把这类文本映射到相近区域。

更准确地说，模型学到的是：

- 相似文本 -> 向量更近
- 不同文本 -> 向量更远

这不是每次都绝对正确，但好的 embedding 模型通常具备这种性质。

### 余弦相似度如何工作

余弦相似度比较的是两个向量的方向是否接近，而不是它们的绝对长度是否一样。

公式：

```text
cos(a, b) = (a · b) / (|a| |b|)
```

可以把两个向量想成两支箭头：

```text
a: ------>
b: ----->
c:    ^
      |
```

- `a` 和 `b` 方向接近，余弦相似度接近 `1`
- `a` 和 `c` 接近垂直，余弦相似度接近 `0`
- 如果两个向量方向完全相反，结果接近 `-1`

在 embedding 检索里，方向越接近，通常表示语义越接近，因此余弦相似度是最常见的相似度计算方式之一。

In [ ]:
import OpenAI from 'openai'
const openai = new OpenAI()


In [ ]:
const embedding = await openai.embeddings.create({
  model: 'text-embedding-3-small',
  input: 'Your text string goes here',
  encoding_format: 'float',
})

console.log('维度', embedding.data[0].embedding.length)
console.log(embedding)


In [ ]:
const embedding2 = await openai.embeddings.create({
  model: 'qwen/qwen3-embedding-8b',
  input: 'Your text string goes here',
  encoding_format: 'float',
})

console.log('维度', embedding2.data[0].embedding.length)
console.log(embedding2)


余弦相似度函数

In [ ]:
/**
 * 计算两个向量的余弦相似度
 * 余弦相似度衡量的是两个向量方向的接近程度，值域为 [-1, 1]
 *   1  表示方向完全相同（语义最相似）
 *   0  表示方向正交（无关联）
 *  -1  表示方向完全相反
 */
function cosineSimilarity(a: number[], b: number[]): number {
  // 维度检查：两个向量长度必须一致才能逐元素相乘
  if (a.length !== b.length) {
    throw new Error('向量维度不一致，不能直接比较')
  }

  let dot = 0 // 点积：a · b = Σ(a[i] * b[i])
  let normA = 0 // 向量 a 的模的平方：|a|² = Σ(a[i]²)
  let normB = 0 // 向量 b 的模的平方：|b|² = Σ(b[i]²)

  // 一次遍历同时计算点积和两个向量的模
  for (let i = 0; i < a.length; i++) {
    dot += a[i] * b[i] // 累加点积
    normA += a[i] * a[i] // 累加 a 的模平方
    normB += b[i] * b[i] // 累加 b 的模平方
  }

  // 余弦相似度公式：cos(a, b) = (a · b) / (|a| * |b|)
  return dot / (Math.sqrt(normA) * Math.sqrt(normB))
}


In [ ]:
const texts = [
  '机器学习入门教程',
  '如何开始学习机器学习',
  '今天天气很好',
]

const items = await Promise.all(
  texts.map(async (text) => {
    const res = await openai.embeddings.create({
      model: 'text-embedding-3-small',
      input: text,
      encoding_format: 'float',
    })

    return {
      text,
      embedding: res.data[0].embedding,
    }
  }),
)

console.log(
  '1 vs 2',
  cosineSimilarity(items[0].embedding, items[1].embedding),
)

console.log(
  '1 vs 3',
  cosineSimilarity(items[0].embedding, items[2].embedding),
)


### OpenRouter 中常见 Embedding 模型对比

下面这张表整理了 OpenRouter 中几种常见的 embedding 模型。

注意：

- 数据口径截至 `2026-04-14`
- 费用单位为 `USD / 1M input tokens`
- 部分模型支持自定义输出维度，表中展示的是默认或最大维度

| 名称                   | Provider   | 上下文长度 |       费用 | 维度 |
| ---------------------- | ---------- | ---------: | ---------: | ---: |
| Qwen3 Embedding 8B     | Qwen       |        32K | $0.01 / 1M | 4096 |
| Qwen3 Embedding 4B     | Qwen       |     32,768 | $0.02 / 1M | 2560 |
| Text Embedding 3 Small | OpenAI     |      8,192 | $0.02 / 1M | 1536 |
| Text Embedding 3 Large | OpenAI     |      8,192 | $0.13 / 1M | 3072 |
| bge-m3                 | BAAI       |      8,192 | $0.01 / 1M | 1024 |
| Gemini Embedding 001   | Google     |        20K | $0.15 / 1M | 3072 |
| Embed V1 4B            | Perplexity |        32K | $0.03 / 1M | 2560 |
| Text Embedding Ada 002 | OpenAI     |      8,192 | $0.10 / 1M | 1536 |

#### 简要观察

- `Qwen3 Embedding 8B` 价格很低，上下文也长，适合先做中文或多语种 RAG baseline
- `text-embedding-3-small` 是很稳的通用入门选择，接入也简单
- `text-embedding-3-large`、`Gemini Embedding 001` 更偏向高效果路线，但费用更高
- `bge-m3` 在开源检索方案里很常见，维度和成本都比较克制
- `text-embedding-ada-002` 更偏 legacy 兼容，不建议新项目优先选